# Setup (Colab + local compatible)

In [ ]:
# Cell 1 – Install dependencies (Colab) / imports (local)

# If you're in Colab, install packages (safe to rerun)
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q ultralytics thop pandas matplotlib scipy opencv-python

import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cv2
import scipy.io as sio  # for .mat files from EgoHands

from ultralytics import YOLO
from thop import profile

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running in Colab:", IN_COLAB)
print("Using device:", device)
print("Torch version:", torch.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Running in Colab: True
Using device: cuda
Torch version: 2.9.0+cu126


# Project paths and dataset config

In [ ]:
# Cell 2 – Paths and high-level configuration

# Root directory for everything
if IN_COLAB:
    PROJ_ROOT = Path("/content/hand_bbox_baseline")
else:
    PROJ_ROOT = Path.cwd() / "hand_bbox_baseline"

RAW_ROOT   = PROJ_ROOT / "data" / "raw"
WORK_ROOT  = PROJ_ROOT / "data" / "work"
YOLO_ROOT  = PROJ_ROOT / "data" / "hands_yolo"  # final YOLO dataset

RAW_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_ROOT.mkdir(parents=True, exist_ok=True)

# EgoHands labeled data archive (pixel-level masks and frames) :contentReference[oaicite:1]{index=1}
EGO_ZIP_URL  = "https://vision.soic.indiana.edu/egohands_files/egohands_data.zip"
EGO_ZIP_PATH = RAW_ROOT / "egohands_data.zip"
EGO_EXTRACT_DIR = WORK_ROOT / "egohands_data"  # where we unzip

print("Project root:", PROJ_ROOT)
print("Raw root:", RAW_ROOT)
print("Work root:", WORK_ROOT)
print("YOLO dataset root:", YOLO_ROOT)


Project root: /content/hand_bbox_baseline
Raw root: /content/hand_bbox_baseline/data/raw
Work root: /content/hand_bbox_baseline/data/work
YOLO dataset root: /content/hand_bbox_baseline/data/hands_yolo


# Download EgoHands (only if not already)

In [ ]:
# Cell 3 – Download EgoHands labeled data (if not already downloaded)

import urllib.request

if not EGO_ZIP_PATH.exists():
    print("Downloading EgoHands labeled data...")
    urllib.request.urlretrieve(EGO_ZIP_URL, EGO_ZIP_PATH)
    print("Downloaded to:", EGO_ZIP_PATH)
else:
    print("EgoHands zip already exists:", EGO_ZIP_PATH)


Downloaded to: /content/hand_bbox_baseline/data/raw/egohands_data.zip


# Unzip EgoHands (only if not already)

In [ ]:
# Cell 4 – Unzip EgoHands labeled data (if not already extracted)

import zipfile

if not EGO_EXTRACT_DIR.exists():
    print("Extracting EgoHands zip...")
    with zipfile.ZipFile(EGO_ZIP_PATH, 'r') as zf:
        zf.extractall(EGO_EXTRACT_DIR)
    print("Extracted to:", EGO_EXTRACT_DIR)
else:
    print("EgoHands already extracted at:", EGO_EXTRACT_DIR)


Extracting EgoHands zip...
Extracted to: /content/hand_bbox_baseline/data/work/egohands_data


# Explore Egohands structure (just to know what we have)

In [ ]:
# Cell 5 – Quick peek at EgoHands folder structure

for root, dirs, files in os.walk(EGO_EXTRACT_DIR):
    print(root)
    print("  dirs:", dirs[:5])
    print("  files:", [f for f in files if f.endswith(('.jpg', '.mat'))][:5])
    break  # just top-level


/content/hand_bbox_baseline/data/work/egohands_data
  dirs: ['_LABELLED_SAMPLES']
  files: ['metadata.mat']


# Convert EgoHands polygons → YOLO boxes

We:

Find all folders that contain a polygons.mat.

For each folder:

  Load polygons.mat (matlab file with polygons per image).
  GitHub

  Sort the .jpg frames.

  For each image, read polygons, compute a bounding box for each hand polygon.

  Create YOLO labels (single class 0 = hand).

In [ ]:
# Cell 6 – Convert EgoHands polygons to YOLO labels (in memory)

from glob import glob

def poly_to_bbox(points):
    """
    points: array of shape (N, 2) with pixel coordinates [x, y]
    returns: (x1, y1, x2, y2) in pixel coordinates
    """
    xs = points[:, 0]
    ys = points[:, 1]
    x1, y1 = xs.min(), ys.min()
    x2, y2 = xs.max(), ys.max()
    return x1, y1, x2, y2


def bbox_to_yolo(x1, y1, x2, y2, img_w, img_h):
    """
    Convert pixel bbox to YOLO normalized cx, cy, w, h.
    """
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return cx, cy, w, h


# Gather all .mat polygons files
poly_files = sorted(glob(str(EGO_EXTRACT_DIR / "**" / "polygons.mat"), recursive=True))
print("Found polygons.mat files:", len(poly_files))

all_samples = []  # list of dicts: {image_path, boxes: [(cx,cy,w,h), ...]}

for poly_path in poly_files:
    poly_path = Path(poly_path)
    folder = poly_path.parent

    # all jpg frames in that folder
    frame_paths = sorted(glob(str(folder / "*.jpg")))
    if not frame_paths:
        continue

    # load matlab polygons
    mat = sio.loadmat(poly_path)
    # 'polygons' is a 1D cell array; polygons[0] is list over images :contentReference[oaicite:3]{index=3}
    polys = mat.get("polygons", None)
    if polys is None:
        print("WARNING: 'polygons' not found in", poly_path)
        continue

    polys = polys[0]  # shape (num_images,)

    if len(polys) != len(frame_paths):
        print(f"WARNING: polygons count {len(polys)} != frames {len(frame_paths)} in {folder}")

    # For safety, iterate over min length
    for img_idx, frame_path in enumerate(frame_paths[:len(polys)]):
        frame_path = Path(frame_path)
        img = cv2.imread(str(frame_path))
        if img is None:
            print("Could not read image:", frame_path)
            continue
        img_h, img_w = img.shape[:2]

        # polys[img_idx] is an array of length #hands, each entry Nx2 or empty
        poly_list = polys[img_idx]
        if poly_list.size == 0:
            # no hands
            continue

        boxes_yolo = []
        for p in poly_list:
            if p.size == 0:
                continue
            # p is Nx2 array
            x1, y1, x2, y2 = poly_to_bbox(p)
            cx, cy, w, h = bbox_to_yolo(x1, y1, x2, y2, img_w, img_h)

            # filter degenerate boxes
            if w <= 0 or h <= 0:
                continue
            # some sanity clamping
            cx = np.clip(cx, 0, 1)
            cy = np.clip(cy, 0, 1)
            w  = np.clip(w, 0, 1)
            h  = np.clip(h, 0, 1)

            boxes_yolo.append((cx, cy, w, h))

        if not boxes_yolo:
            continue

        all_samples.append({
            "image_path": frame_path,
            "img_w": img_w,
            "img_h": img_h,
            "boxes": boxes_yolo
        })

print("Total labeled images with at least one hand:", len(all_samples))


Found polygons.mat files: 48
Total labeled images with at least one hand: 4787


# Train/val/test split and write YOLO structure

In [ ]:
# Cell 7 – Split into train/val/test and write YOLO dataset

from shutil import copy2
import random
import yaml

random.seed(42)

# Shuffle and split (80% train, 10% val, 10% test)
random.shuffle(all_samples)
n_total = len(all_samples)
n_train = int(0.8 * n_total)
n_val   = int(0.1 * n_total)
n_test  = n_total - n_train - n_val

train_samples = all_samples[:n_train]
val_samples   = all_samples[n_train:n_train+n_val]
test_samples  = all_samples[n_train+n_val:]

print(f"Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")

# Prepare folders
for split in ["train", "val", "test"]:
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)


def write_split(samples, split):
    img_out_dir = YOLO_ROOT / "images" / split
    lbl_out_dir = YOLO_ROOT / "labels" / split

    for s in samples:
        src_img_path = s["image_path"]
        dst_img_path = img_out_dir / src_img_path.name
        copy2(src_img_path, dst_img_path)

        # YOLO label file
        dst_lbl_path = lbl_out_dir / (src_img_path.stem + ".txt")
        with open(dst_lbl_path, "w") as f:
            for (cx, cy, w, h) in s["boxes"]:
                # single class "hand" -> class id = 0
                f.write(f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")


write_split(train_samples, "train")
write_split(val_samples,   "val")
write_split(test_samples,  "test")

print("YOLO dataset written to:", YOLO_ROOT)


Train: 3829, Val: 478, Test: 480
YOLO dataset written to: /content/hand_bbox_baseline/data/hands_yolo


In [ ]:
# Still Cell 7 – Create data.yaml for YOLO

data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "hand"},
}

DATA_YAML_PATH = YOLO_ROOT / "data.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f)

print("Created data.yaml at:", DATA_YAML_PATH)


Created data.yaml at: /content/hand_bbox_baseline/data/hands_yolo/data.yaml


# Common training configuration + experiment list

In [9]:
# Cell 8 – Experiments configuration (YOLOv8n with different LRs)

DATA_YAML = str(DATA_YAML_PATH)

# Where YOLO training runs will be saved
PROJECT_ROOT = PROJ_ROOT / "runs_yolo"
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# Common hyperparameters
IMG_SIZE   = 416
BATCH_SIZE = 16  # adjust for your GPU
EPOCHS     = 100

BASE_LR = 1e-3
LOW_LR  = 1e-4  # teacher's "very low" LR

EXPERIMENTS = [
    {
        "name": "yolov8n_base_lr",
        "model_path": "yolov8n.pt",  # pretrained weights
        "lr0": BASE_LR,
    },
    {
        "name": "yolov8n_low_lr",
        "model_path": "yolov8n.pt",
        "lr0": LOW_LR,
    },
]

print("Dataset yaml:", DATA_YAML)
print("Experiments:", [e["name"] for e in EXPERIMENTS])


Dataset yaml: /content/hand_bbox_baseline/data/hands_yolo/data.yaml
Experiments: ['yolov8n_base_lr', 'yolov8n_low_lr']


# Train YOLOv8n for each experiment

In [11]:
# Cell 9 – Training helper and running experiments

def train_experiment(exp_cfg):
    """
    Train a YOLOv8 model for a given experiment configuration.
    """
    name = exp_cfg["name"]
    model_path = exp_cfg["model_path"]
    lr0 = exp_cfg["lr0"]

    print(f"\n=== Training experiment: {name} ===")
    print(f"Model: {model_path} | lr0={lr0} | epochs={EPOCHS} | batch={BATCH_SIZE} | imgsz={IMG_SIZE}")

    model = YOLO(model_path)

    results = model.train(
        data=DATA_YAML,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        lr0=lr0,
        optimizer="AdamW",
        momentum=0.9,
        patience=25,  # stop if no val improvement for 25 epochs
        project=str(PROJECT_ROOT),
        name=name,
        exist_ok=True,
        verbose=True
    )

    run_dir = Path(results.save_dir)
    print("Run directory:", run_dir)
    return model, run_dir


trained_models = {}
run_dirs = {}

for exp in EXPERIMENTS:
    model, run_dir = train_experiment(exp)
    trained_models[exp["name"]] = model
    run_dirs[exp["name"]] = run_dir



=== Training experiment: yolov8n_base_lr ===
Model: yolov8n.pt | lr0=0.001 | epochs=100 | batch=16 | imgsz=416
Ultralytics 8.3.233 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/hand_bbox_baseline/data/hands_yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.9, mosaic=1.0, multi_scale=False, name=yo

KeyboardInterrupt: 

# Utilities: box conversion + IoU + greedy matching

In [ ]:
# Cell 10 – Box utilities: convert YOLO normalized and IoU

def yolo_xywhn_to_xyxyn(cx, cy, w, h):
    x1 = cx - w / 2.0
    y1 = cy - h / 2.0
    x2 = cx + w / 2.0
    y2 = cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=np.float32)


def box_iou_xyxy(box_a, box_b):
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - inter
    if union <= 0.0:
        return 0.0
    return inter / union


def greedy_match_boxes(pred_boxes, gt_boxes, iou_min=0.1):
    """
    Greedy match PM ↔ GT based on IoU.
    """
    matches = []
    if len(pred_boxes) == 0 or len(gt_boxes) == 0:
        return matches

    pred_used = set()
    gt_used = set()

    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)), dtype=np.float32)
    for i, pb in enumerate(pred_boxes):
        for j, gb in enumerate(gt_boxes):
            iou_matrix[i, j] = box_iou_xyxy(pb, gb)

    while True:
        i, j = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)
        best_iou = iou_matrix[i, j]
        if best_iou < iou_min:
            break
        if i in pred_used or j in gt_used:
            iou_matrix[i, j] = -1.0
            continue

        matches.append((i, j, best_iou))
        pred_used.add(i)
        gt_used.add(j)
        iou_matrix[i, :] = -1.0
        iou_matrix[:, j] = -1.0

    return matches


# Load GT boxes from YOLO val labels

In [ ]:
# Cell 11 – Load GT boxes (normalized xyxy) for a given image

VAL_IMAGES_DIR = YOLO_ROOT / "images" / "val"
VAL_LABELS_DIR = YOLO_ROOT / "labels" / "val"

def load_gt_boxes_for_image(image_path, labels_dir=VAL_LABELS_DIR):
    image_path = Path(image_path)
    label_path = Path(labels_dir) / (image_path.stem + ".txt")
    if not label_path.exists():
        return []

    gt_boxes = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, w, h = parts
            cx, cy, w, h = map(float, (cx, cy, w, h))
            box_xyxy = yolo_xywhn_to_xyxyn(cx, cy, w, h)
            gt_boxes.append(box_xyxy)
    return gt_boxes


# Extra metrics: IoU, MAE, MSE, R²

In [ ]:
# Cell 12 – Evaluate extra metrics on validation set

def evaluate_extra_metrics(model, conf_threshold=0.25, max_images=None):
    all_gt_coords = []
    all_pred_coords = []
    all_ious = []

    results_gen = model.predict(
        source=str(VAL_IMAGES_DIR),
        imgsz=IMG_SIZE,
        conf=conf_threshold,
        iou=0.5,
        stream=True,
        verbose=False
    )

    for idx, r in enumerate(results_gen):
        if max_images is not None and idx >= max_images:
            break

        img_path = r.path
        if r.boxes is None or r.boxes.xyxy.shape[0] == 0:
            continue

        pred_xyxy_px = r.boxes.xyxy.cpu().numpy()
        h, w = r.orig_shape

        pred_boxes_norm = []
        for bx in pred_xyxy_px:
            x1, y1, x2, y2 = bx
            pred_boxes_norm.append(
                np.array([x1 / w, y1 / h, x2 / w, y2 / h], dtype=np.float32)
            )

        gt_boxes_norm = load_gt_boxes_for_image(img_path)

        matches = greedy_match_boxes(pred_boxes_norm, gt_boxes_norm, iou_min=0.1)

        for (pi, gi, iou) in matches:
            pb = pred_boxes_norm[pi]
            gb = gt_boxes_norm[gi]
            all_ious.append(iou)
            all_pred_coords.append(pb)
            all_gt_coords.append(gb)

    if len(all_gt_coords) == 0:
        print("No matches found; check thresholds/paths.")
        return None

    all_gt = np.stack(all_gt_coords, axis=0)
    all_pred = np.stack(all_pred_coords, axis=0)
    all_ious = np.array(all_ious)

    mae = np.mean(np.abs(all_pred - all_gt))
    mse = np.mean((all_pred - all_gt)**2)

    y_true = all_gt.reshape(-1)
    y_pred = all_pred.reshape(-1)
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    metrics = {
        "mean_iou": float(np.mean(all_ious)),
        "mae": float(mae),
        "mse": float(mse),
        "r2": float(r2),
        "num_matches": len(all_gt_coords),
    }
    return metrics


extra_metrics = {}
for exp in EXPERIMENTS:
    name = exp["name"]
    model = trained_models[name]
    print(f"\n=== Extra metrics for {name} ===")
    m = evaluate_extra_metrics(model, conf_threshold=0.25, max_images=None)
    print(m)
    extra_metrics[name] = m


# Training curves (loss & mAP)

In [ ]:
# Cell 13 – Plot training curves from results.csv

def plot_training_curves(run_dir, title=None):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        print("No results.csv at", csv_path)
        return

    df = pd.read_csv(csv_path)
    epochs = df["epoch"]

    plt.figure(figsize=(10, 6))

    if "train/box_loss" in df.columns:
        plt.plot(epochs, df["train/box_loss"], label="train box loss")
    if "val/box_loss" in df.columns:
        plt.plot(epochs, df["val/box_loss"], label="val box loss")
    if "metrics/mAP50(B)" in df.columns:
        plt.plot(epochs, df["metrics/mAP50(B)"], label="mAP50(B)")
    if "metrics/mAP50-95(B)" in df.columns:
        plt.plot(epochs, df["metrics/mAP50-95(B)"], label="mAP50-95(B)")

    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True)
    plt.title(title if title else str(run_dir))
    plt.show()


for exp in EXPERIMENTS:
    name = exp["name"]
    run_dir = run_dirs[name]
    print(f"\n=== Curves for {name} ===")
    plot_training_curves(run_dir, title=name)


# Params, FLOPs, latency CPU/GPU

In [ ]:
# Cell 14 – Params, FLOPs (MACs), latency

def count_trainable_params(model):
    return sum(p.numel() for p in model.model.parameters() if p.requires_grad)


def compute_flops(model, img_size=IMG_SIZE):
    dummy = torch.randn(1, 3, img_size, img_size).to(device)
    model.model.to(device)
    model.model.eval()
    flops, params = profile(model.model, inputs=(dummy,), verbose=False)
    return flops, params


def benchmark_latency(model, img_size=IMG_SIZE, device_str="cuda", n_warmup=10, n_iters=50):
    dev = torch.device(device_str)
    model.model.to(dev)
    model.model.eval()

    dummy = torch.randn(1, 3, img_size, img_size, device=dev)

    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model.model(dummy)
        if device_str == "cuda":
            torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_iters):
            t0 = time.perf_counter()
            _ = model.model(dummy)
            if device_str == "cuda":
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            times.append(t1 - t0)

    return np.mean(times) * 1000.0  # ms


model_stats = {}

for exp in EXPERIMENTS:
    name = exp["name"]
    model = trained_models[name]
    print(f"\n=== Model stats for {name} ===")

    n_params = count_trainable_params(model)
    params_m = n_params / 1e6
    print(f"Params: {params_m:.3f} M")

    try:
        flops, params_thop = compute_flops(model, IMG_SIZE)
        flops_g = flops / 1e9
        print(f"Approx MACs: {flops_g:.3f} G")
    except Exception as e:
        print("FLOPs estimation failed:", e)
        flops_g = None

    gpu_ms = None
    if torch.cuda.is_available():
        gpu_ms = benchmark_latency(model, IMG_SIZE, device_str="cuda")
        print(f"GPU latency: {gpu_ms:.2f} ms / img")

    cpu_ms = benchmark_latency(model, IMG_SIZE, device_str="cpu")
    print(f"CPU latency: {cpu_ms:.2f} ms / img")

    model_stats[name] = {
        "params_m": params_m,
        "flops_gmacs": flops_g,
        "gpu_ms": gpu_ms,
        "cpu_ms": cpu_ms,
    }


# Final comparison table (for TFG)

In [ ]:
# Cell 15 – Build and export comparison table

rows = []

for exp in EXPERIMENTS:
    name = exp["name"]
    stats = model_stats.get(name, {})
    extra = extra_metrics.get(name, {})

    run_dir = run_dirs[name]
    csv_path = Path(run_dir) / "results.csv"
    map50 = None
    map5095 = None
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        if "metrics/mAP50(B)" in df.columns:
            map50 = df["metrics/mAP50(B)"].iloc[-1]
        if "metrics/mAP50-95(B)" in df.columns:
            map5095 = df["metrics/mAP50-95(B)"].iloc[-1]

    row = {
        "experiment": name,
        "params_M": stats.get("params_m"),
        "FLOPs_GMACs": stats.get("flops_gmacs"),
        "GPU_ms_per_img": stats.get("gpu_ms"),
        "CPU_ms_per_img": stats.get("cpu_ms"),
        "mAP50": map50,
        "mAP50_95": map5095,
        "mean_IoU": extra.get("mean_iou"),
        "MAE_coords": extra.get("mae"),
        "MSE_coords": extra.get("mse"),
        "R2_coords": extra.get("r2"),
        "num_matches": extra.get("num_matches"),
    }
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary


In [ ]:
# Still Cell 15 – Save to CSV
summary_path = PROJ_ROOT / "hands_model_comparison.csv"
df_summary.to_csv(summary_path, index=False)
print("Saved comparison table to:", summary_path)
